In [1]:
import numpy as np
import os
import scipy.sparse as sp

In [47]:
def _load_facebook_data(dataset_dir, ego_id):
    """Load a graph from raw facebook files in the dataset directory."""
    feat_path = os.path.join(dataset_dir, f"{ego_id}.feat")
    egofeat_path = os.path.join(dataset_dir, f"{ego_id}.egofeat")
    edge_path = os.path.join(dataset_dir, f"{ego_id}.edges")
    circle_path = os.path.join(dataset_dir, f"{ego_id}.circles")

    id_map = {}
    all_features = []

    # 1. Load Alter Nodes Features
    if os.path.exists(feat_path):
        with open(feat_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                node_id = int(parts[0])
                feats = np.array([float(x) for x in parts[1:]], dtype=np.float32)
                
                if node_id not in id_map:
                    id_map[node_id] = len(id_map)
                    all_features.append(feats)

    if not all_features:
        if os.path.exists(egofeat_path):
             pass # Add ego features here if desired
        if not all_features:
             raise ValueError(f"No feature data found for ego {ego_id}")
    
    X = np.array(all_features)
    num_nodes = len(id_map)

    # 2. Build Adjacency Matrix
    adj_set = set()
    if os.path.exists(edge_path):
        with open(edge_path, 'r') as f:
            for line in f:
                u, v = map(int, line.strip().split())
                if u in id_map and v in id_map:
                    u_idx, v_idx = id_map[u], id_map[v]
                    adj_set.add((u_idx, v_idx))
                    adj_set.add((v_idx, u_idx))

    rows, cols = zip(*adj_set) if adj_set else ([], [])
    data = np.ones(len(rows)) 
    A = sp.csr_matrix((data, (rows, cols)), shape=(num_nodes, num_nodes))

    # 3. Labels (Circles) - GROUND TRUTH FORMAT (List of Sets)
    ground_truth_communities = []
    
    if os.path.exists(circle_path):
        with open(circle_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                # Format: circle<id> node1 node2 ...
                circle_nodes = set()
                for node_str in parts[1:]:
                    node_id = int(node_str)
                    if node_id in id_map:
                        circle_nodes.add(id_map[node_id])
                
                if circle_nodes:
                    ground_truth_communities.append(circle_nodes)

    return X, ground_truth_communities, A

In [48]:
X, label, A = _load_facebook_data("D:/FYP/RGC/dataset/facebook", "348")

In [50]:
print("Feature shape:", X.shape)
print("Adjacency shape:", A.shape)

Feature shape: (227, 161)
Adjacency shape: (227, 227)


In [53]:
#count number of nodes in each community
for i, community in enumerate(label):
    print(f"Community {i}: {len(community)} nodes")


Community 0: 20 nodes
Community 1: 201 nodes
Community 2: 25 nodes
Community 3: 5 nodes
Community 4: 9 nodes
Community 5: 21 nodes
Community 6: 12 nodes
Community 7: 18 nodes
Community 8: 41 nodes
Community 9: 13 nodes
Community 10: 4 nodes
Community 11: 117 nodes
Community 12: 9 nodes
Community 13: 72 nodes


In [46]:
print(list(A))

[<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (1, 227)>, <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20 stored elements and shape (1, 227)>, <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (1, 227)>, <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 22 stored elements and shape (1, 227)>, <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 52 stored elements and shape (1, 227)>, <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 25 stored elements and shape (1, 227)>, <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 38 stored elements and shape (1, 227)>, <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (1, 227)>, <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 17 stored elements and shape (1, 227)>, <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 0 stored elements and 

In [45]:
#count number of edges
print("Number of edges:", A.nnz)

Number of edges: 6384
